In [2]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from dig4bio.io import read_raman_file
from sklearn.linear_model import LinearRegression
from dig4bio.constants import FINGERPRINT_COLUMNS, LABEL_COLUMNS
from dig4bio.cv import augmented_cross_validate_sample_folds
from dig4bio.models import create_model_factory, AveragePredictor
from sklearn.model_selection import cross_validate, GroupKFold

In [3]:
source_df = read_raman_file(level='processed',subfolder='source_grid_fingerprint_linear',name='source_datasets.csv')
source_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
560,4255.953500,4235.632400,4214.724433,4196.636100,4184.305633,4177.960333,4175.489000,4173.535267,4169.326533,4161.863200,...,585.761701,587.678529,586.446932,580.792104,0.53269,0.68172,0.056479,0.0,0,kaiser
2075,40376.176284,40133.956092,39927.996815,39787.835120,39661.025834,39476.616320,39260.689047,39102.798270,39021.404143,38938.232639,...,3926.161902,3974.997980,4013.667123,4012.356257,2.01263,0.00000,0.004711,0.0,2,tornado
1535,2364.371816,2377.699671,2340.571013,2288.892088,2229.229614,2100.689029,2107.597405,2171.663256,2287.512556,2547.358496,...,2732.928468,2736.814862,2728.545202,2721.740653,10.71080,0.74408,3.480790,0.0,2,tec
315,1052.180000,1049.100000,1046.020000,1043.410000,1040.800000,1036.965000,1033.130000,1036.300000,1039.470000,1033.475000,...,795.050000,800.690000,797.030000,793.370000,4.78962,1.03913,1.497530,0.0,0,anton785
430,758.830000,754.675000,750.520000,748.095000,745.670000,745.115000,744.560000,745.870000,747.180000,747.730000,...,747.175000,750.350000,751.125000,751.900000,10.39640,0.68902,1.602520,0.0,3,anton785
852,7882.257511,7852.590517,7834.056034,7815.521552,7760.549356,7697.030043,7680.758621,7715.672414,7737.789700,7692.725322,...,1617.520468,1606.117647,1585.529412,1600.614035,0.00000,0.65286,0.046091,0.0,2,metrohm
1304,7044.795392,7045.607991,7031.466529,6990.274855,6940.232609,6901.226351,6891.153803,6919.098533,6952.101109,6949.534597,...,877.263055,867.212095,844.077437,826.598179,0.51645,1.02701,1.328620,0.0,4,mettler
1075,10805.611990,10833.851000,10836.750849,10827.247540,10823.522193,10845.343015,10915.561036,11047.384692,11203.769498,11339.125152,...,2228.858042,2220.878948,2280.102742,2315.522855,5.24672,0.02440,3.361270,0.0,0,mettler
850,7678.000000,7678.616379,7683.357759,7688.099138,7649.716738,7601.218884,7565.241379,7542.827586,7516.377682,7468.738197,...,1618.953216,1606.247059,1588.011765,1576.701754,0.00000,0.65286,0.046091,0.0,2,metrohm
959,7219.180258,7166.659483,7110.193966,7053.728448,7060.381974,7081.841202,7072.103448,7028.568966,6986.987124,6955.656652,...,1583.128655,1568.576471,1542.694118,1547.947368,0.97576,0.73799,0.004316,0.0,3,metrohm


In [ ]:
transfer_plate_df = read_raman_file(level='processed',subfolder='transfer_grid_fingerprint_linear',name='transfer_plate.csv')
transfer_plate_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
111,9531.8125,9458.677019,9475.447205,9478.18125,9466.30625,9368.813665,9290.475,9288.6,9324.322981,9371.15625,...,1657.118012,1685.068323,1706.37500,1718.87500,1712.236025,1694.2375,sample56,2.948548,0.433675,0.634969
76,9713.8125,9605.298137,9628.279503,9603.25000,9528.25000,9489.596273,9453.275,9411.4,9468.534161,9560.01250,...,1655.440994,1691.465839,1698.29375,1666.41875,1687.807453,1699.9375,sample39,11.328211,0.324832,0.942181
153,9470.4375,9388.080745,9458.888199,9477.23750,9440.98750,9352.975155,9289.200,9304.2,9352.503106,9406.61875,...,1669.925466,1668.062112,1672.37500,1684.87500,1685.968944,1671.3875,sample77,2.717046,0.647647,0.581811
104,9665.0625,9613.515528,9688.670807,9682.59375,9591.96875,9546.621118,9514.325,9501.2,9530.366460,9570.88125,...,1669.118012,1697.068323,1700.36875,1670.99375,1696.931677,1712.1125,sample53,11.887974,0.276526,0.487412
61,9653.4375,9619.055901,9629.614907,9620.30000,9590.30000,9519.006211,9451.925,9408.8,9457.086957,9540.29375,...,1668.869565,1690.608696,1701.11875,1696.74375,1715.074534,1716.2250,sample31,8.757543,0.409188,2.076763
90,9553.6875,9544.863354,9563.496894,9515.42500,9397.92500,9354.043478,9336.875,9365.0,9384.981366,9392.95625,...,1685.770186,1705.024845,1714.11875,1709.74375,1720.857143,1716.0500,sample46,1.993671,1.370442,1.366220
12,9385.6875,9370.968944,9420.658385,9391.79375,9281.16875,9181.105590,9125.575,9181.2,9200.689441,9200.81250,...,1663.093168,1690.422360,1702.77500,1695.27500,1683.173913,1665.8500,sample7,7.819112,0.846058,0.785458
4,9575.5000,9552.875776,9551.633540,9542.73125,9525.85625,9452.689441,9393.950,9390.2,9395.757764,9401.98125,...,1708.850932,1705.124224,1704.61250,1708.36250,1719.826087,1713.3625,sample3,3.953448,1.350473,2.132459
161,9584.3750,9550.708075,9517.788820,9505.90000,9515.90000,9441.819876,9369.975,9330.6,9377.130435,9455.24375,...,1645.639752,1686.633540,1703.28125,1687.65625,1683.968944,1675.7125,sample81,3.984727,1.353590,1.999678
178,3155.8750,3142.788820,3180.677019,3182.23750,3145.98750,3108.832298,3084.625,3094.0,3121.074534,3151.02500,...,1292.968944,1317.192547,1335.83750,1347.08750,1325.801242,1298.9125,sample90,2.371494,1.454220,1.969093


In [ ]:
### Linear Sources: Linear regression over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(LinearRegression())

sources_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_lr_errors['devices']))
print('Overall')
display(pd.Series(sources_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-9.814037,-11.112856,-268.714326,-520.361891,-178.036716,-11937.375305,-3.594700,-2182.403212
Na_acetate,-13.009612,-0.868218,-17.183729,-67.974554,-209.363922,-6417.786683,-0.516098,-1015.802629
Mg_SO4,-5.064514,-0.036047,-9.014977,-95.357735,-259.089710,-360.502686,-0.240201,-957.250087


Overall


glucose      -2201.480448
Na_acetate   -1198.156705
Mg_SO4        -277.009218
dtype: float64

In [ ]:
### Averages Sources: Analyte averages over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(AveragePredictor())

sources_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_avgs_errors['devices']))
print('Overall')
display(pd.Series(sources_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.007701,-0.008376,-0.004897,-0.004898,0.003036,-0.000517,-0.007763,-0.020530
Na_acetate,-0.027002,-0.022158,-0.053501,0.000620,-0.026815,-0.003065,-0.053188,-0.003543
Mg_SO4,-0.076486,-0.077122,-0.237535,-0.023226,-0.065905,-0.022300,-0.258090,-0.014577


Overall


glucose      -0.003056
Na_acetate   -0.003696
Mg_SO4       -0.020624
dtype: float64

In [7]:
### Linear Target: Linear regression train + test over the target device only ###

model_factory = create_model_factory(LinearRegression())

target_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_lr_errors['devices']))
print('Overall')
display(pd.Series(target_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,0.515546,0.528299,0.882741,-3.019211,0.907405,0.612470,0.517172,0.121881
Na_acetate,-0.505592,0.778544,0.821707,0.482408,0.556746,-2.572058,0.690904,-1.108420
Mg_SO4,0.549873,0.835158,0.847339,0.614719,0.879381,-2.023251,0.694120,-10.158555


Overall


glucose       0.028888
Na_acetate   -0.231723
Mg_SO4       -1.627209
dtype: float64

In [8]:
### Linear Averages: Analyte Averages train + test over the target device only ###

model_factory = create_model_factory(AveragePredictor())

target_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_avgs_errors['devices']))
print('Overall')
display(pd.Series(target_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.034112,-0.073546,-0.008200,-0.035465,-0.053714,-0.018685,-0.021394,-0.008342
Na_acetate,-0.056263,-0.023727,-0.027337,-0.020718,-0.014491,-0.030386,-0.007855,-0.022776
Mg_SO4,-0.036037,-0.067311,-0.020365,-0.039999,-0.043428,-0.017934,-0.026028,-0.034934


Overall


glucose      -0.032043
Na_acetate   -0.008138
Mg_SO4        0.000577
dtype: float64

In [15]:
model = LinearRegression()

test_df = read_raman_file(name='96_samples',level='processed',subfolder='96samples_grid_fingerprint_linear')

X = transfer_plate_df[FINGERPRINT_COLUMNS]
y = transfer_plate_df[LABEL_COLUMNS]

model.fit(X,y)
test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])

display(test_df)

output_df = (
    test_df.copy()
    .groupby(by='sample')
    .mean()
    .reset_index()
)[LABEL_COLUMNS]

output_df.index.name = 'ID'
output_df = output_df.reset_index()
output_df['ID'] +=1
output_df.to_csv('runs/output/exp_001_baselines.csv',index=False)
output_df

# Public Kaggle score of this: -1.23611

C:\Users\J_Bra\AppData\Local\Temp\ipykernel_7076\2009297296.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])


,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
0,9459.1250,9440.062112,9440.683230,9403.02500,9325.52500,9295.440994,9269.600,9239.6,9269.279503,9315.65000,...,1650.819876,1671.316770,1689.71875,1705.34375,1696.142857,1672.2125,sample1,1.461898,0.685178,0.353928
1,9441.6250,9411.422360,9435.645963,9429.93125,9393.05625,9312.714286,9256.075,9274.2,9299.850932,9323.63125,...,1655.919255,1678.900621,1679.10000,1649.10000,1665.136646,1674.9625,sample1,2.351013,0.583927,0.467618
2,8607.1875,8614.714286,8571.857143,8517.23125,8450.35625,8408.726708,8371.900,8339.4,8352.229814,8381.90000,...,1603.565217,1642.695652,1666.07500,1668.57500,1651.987578,1625.0000,sample2,5.581757,1.732026,1.431091
3,8560.0625,8553.944099,8543.385093,8527.58750,8506.33750,8434.900621,8377.950,8374.2,8410.590062,8458.28125,...,1609.695652,1627.086957,1630.55000,1615.55000,1632.074534,1635.8375,sample2,6.211434,1.766747,1.229222
4,9432.6875,9347.453416,9321.987578,9321.55625,9347.18125,9274.819876,9216.275,9224.4,9251.919255,9279.65625,...,1677.118012,1705.068323,1704.33750,1665.58750,1693.024845,1707.9500,sample3,4.254881,0.494294,0.999877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,9424.8750,9449.583851,9415.422360,9380.85000,9345.85000,9276.770186,9215.950,9187.2,9226.993789,9288.96875,...,1638.571429,1652.857143,1652.13125,1631.50625,1642.950311,1645.8625,sample94,3.863245,0.210255,0.701643
188,9286.9375,9219.875776,9218.633540,9176.35000,9091.35000,9072.602484,9057.475,9030.6,9054.521739,9094.01250,...,1625.894410,1648.254658,1663.95625,1670.83125,1652.925466,1624.9625,sample95,2.050828,1.525044,0.744649
189,9425.3750,9358.006211,9348.068323,9301.35000,9216.35000,9203.683230,9197.850,9186.6,9245.931677,9329.59375,...,1638.242236,1669.298137,1678.40000,1658.40000,1670.981366,1669.5000,sample95,1.719697,1.459425,0.542773
190,9433.0625,9367.043478,9297.478261,9233.82500,9176.32500,9134.068323,9090.300,9037.8,9117.720497,9246.71875,...,1609.689441,1651.925466,1668.74375,1651.86875,1650.546584,1638.9375,sample96,4.173476,1.753767,0.483940


,ID,glucose,Na_acetate,Mg_SO4
0,1,1.906455,0.634553,0.410773
1,2,9.145176,0.833975,0.326851
2,3,4.515469,1.227353,1.198945
3,4,0.097181,0.947006,0.940205
4,5,0.501357,1.248227,1.221857
...,...,...,...,...
91,92,0.654263,1.251363,1.186624
92,93,4.352456,0.247024,0.550650
93,94,3.855683,0.297089,0.666311
94,95,1.885262,1.492234,0.643711


In [ ]:
comparison = pd.DataFrame(
    (sources_avgs_errors['total'],
    sources_lr_errors['total'],
    target_avgs_errors['total'],
    target_lr_errors['total']),
).T

comparison.columns = ['Averages Sources', 'Linear Sources', 'Averages Target', 'Linear Target']
comparison

,Samples Averages,Samples Linear,Target Averages,Target Linear
glucose,-0.003056,-2201.480448,-0.032043,0.028888
Na_acetate,-0.003696,-1198.156705,-0.008138,-0.231723
Mg_SO4,-0.020624,-277.009218,0.000577,-1.627209


In [17]:
comparison.mean(axis=0)

Samples Averages      -0.009125
Samples Linear     -1225.548790
Target Averages       -0.013202
Target Linear         -0.610015
dtype: float64